# 3 · Inverted indexes and the SINTER tightening

`full_realtime` paid most of its cost fetching every campaign. The first
optimisation is to *not* fetch campaigns that obviously don't match the
user. Build inverted-index sets keyed by user attribute, then ask Redis
for the intersection.

This notebook walks two SINTER plans:

- `maid_bruteforce_sinter` — a 26-probe plan, each probe its own round trip;
- `maid_tightened_sinter` — a 3-probe plan, all in one pipelined round trip.

The tightening collapses the round-trip count from ~28 to ~3. That is the
single biggest reason this mode is faster than the brute-force one — not
the number of probes, the number of *round trips* to issue them.


In [1]:
# Locate the repo root so `notebooks._demo_setup` is importable regardless
# of where the kernel was launched (the package layout requires the repo
# root on sys.path).
import sys
from pathlib import Path
_repo_root = Path.cwd().resolve()
while _repo_root != _repo_root.parent and not (_repo_root / "pyproject.toml").exists():
    _repo_root = _repo_root.parent
if str(_repo_root) not in sys.path:
    sys.path.insert(0, str(_repo_root))

from notebooks._demo_setup import connect_redis, StepTimer
client = connect_redis()

connected to redis://localhost:6381/0
  users=4000  campaigns=2500  precompute_version=v17_2500_12


## What an inverted index looks like

`idx:geo:<country>` is a Redis SET containing the campaign IDs whose
targeting includes that country. Same for `idx:state:`, `idx:device:`,
`idx:device_type:`, `idx:card_tier:`, and `idx:segment:`.


In [2]:
for key in ['idx:geo:US', 'idx:device:iOS', 'idx:card_tier:Gold',
            'idx:segment:travel_high', 'idx:segment:gaming_high']:
    members = client.smembers(key)
    print(f'  {key:<28} {len(members):>5} campaigns; sample: {sorted(members)[:3]}')

  idx:geo:US                    1216 campaigns; sample: ['c00000', 'c00003', 'c00006']
  idx:device:iOS                1564 campaigns; sample: ['c00000', 'c00003', 'c00005']
  idx:card_tier:Gold            1491 campaigns; sample: ['c00000', 'c00001', 'c00003']
  idx:segment:travel_high        226 campaigns; sample: ['c00012', 'c00033', 'c00036']
  idx:segment:gaming_high        244 campaigns; sample: ['c00003', 'c00006', 'c00033']


## A sample MAID

Use the same MAID across all three notebooks (3 → 4 → 5) so the comparison
is apples-to-apples.


In [3]:
from app.models import UserProfile
profile = client.hgetall('maid:maid_00042')
user = UserProfile.from_redis_hash(profile)
print(f'maid_id     = {user.user_id}')
print(f'geo / state = {user.geo} / {user.state}')
print(f'device      = {user.device_type} / {user.device}')
print(f'card_tier   = {user.card_tier}')
print(f'segments    = {user.segments[:6]}')

maid_id     = maid_00042
geo / state = US / IL
device      = ctv / Roku
card_tier   = Standard
segments    = ['fitness_high', 'finance_medium', 'gaming_medium', 'streaming_medium', 'tech_medium']


## The bruteforce 26-probe plan

The legacy planner explores combinations of geo, state, device, device_type,
card_tier, and the user's strong segments. Each probe is a separate
`SINTER` call and each call is a separate Redis round trip.


In [4]:
from app.candidate import build_legacy_union_probe_candidate_lookup_keys
legacy_probes = build_legacy_union_probe_candidate_lookup_keys(user, strong_signal_count=2)
print(f'probe count = {len(legacy_probes)}')
print()
print('first 6 probes:')
for keys in legacy_probes[:6]:
    print(f'  SINTER ' + ' '.join(keys))
print('  ...')
print(f'last probe:')
print(f'  SINTER ' + ' '.join(legacy_probes[-1]))

probe count = 26

first 6 probes:
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:device:Roku idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:device:Roku idx:segment:finance_medium
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:device_type:ctv idx:segment:finance_medium
  SINTER idx:card_tier:Standard idx:geo:US idx:device:Roku idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:device:Roku idx:segment:finance_medium
  ...
last probe:
  SINTER idx:geo:US


Run the bruteforce plan one round trip at a time, the way the
legacy code path does it.


In [5]:
timer = StepTimer()

with timer.step('legacy_26_probes_sequential'):
    probe_results = []
    for keys in legacy_probes:
        result = client.sinter(keys)   # one Redis round trip per probe
        if result:
            probe_results.append(sorted(result))

unique_candidates = set()
for batch in probe_results:
    unique_candidates.update(batch)

print(f'unique campaign IDs returned: {len(unique_candidates)}')
print(f'round trips:                  {len(legacy_probes)}')
print()
print(timer.summary())

unique campaign IDs returned: 1453
round trips:                  26

     legacy_26_probes_sequential    8.579 ms
--------------------------------------------
                           TOTAL    8.579 ms


## The tightened 3-probe plan

The tightened planner notices that 26 probes were not actually buying us
much recall. The compact plan keeps:

- one probe per strong segment (with the strict geo + device + card_tier base),
- plus one strict-base fallback probe.

That's typically `3` probes for a 2-strong-segment user. And — the bigger
win — they are issued in a single pipeline, so it costs **one** Redis
round trip instead of N.


In [6]:
from app.candidate import build_union_probe_candidate_lookup_keys
tight_probes = build_union_probe_candidate_lookup_keys(user, strong_signal_count=2)
print(f'probe count = {len(tight_probes)}')
print()
for keys in tight_probes:
    print(f'  SINTER ' + ' '.join(keys))

probe count = 3

  SINTER idx:card_tier:Standard idx:geo:US idx:state:IL idx:device_type:ctv idx:device:Roku idx:segment:fitness_high
  SINTER idx:card_tier:Standard idx:geo:US idx:state:IL idx:device_type:ctv idx:device:Roku idx:segment:finance_medium
  SINTER idx:card_tier:Standard idx:geo:US idx:state:IL idx:device_type:ctv idx:device:Roku


In [7]:
with timer.step('tightened_3_probes_pipelined'):
    pipe = client.pipeline(transaction=False)
    for keys in tight_probes:
        pipe.sinter(keys)
    pipelined_results = pipe.execute()       # one round trip total

unique_tightened = set()
for batch in pipelined_results:
    unique_tightened.update(batch)

print(f'unique campaign IDs returned: {len(unique_tightened)}')
print(f'round trips:                  1   (pipelined)')
print()
print(timer.summary())

unique campaign IDs returned: 190
round trips:                  1   (pipelined)

     legacy_26_probes_sequential    8.579 ms
    tightened_3_probes_pipelined    0.986 ms
--------------------------------------------
                           TOTAL    9.565 ms


## What the timing tells us

Two effects compound here:

1. The probe count drops from ~26 to 3, so Redis does less SET algebra.
2. The round-trip count drops from ~26 to 1 because the tightened plan
   pipelines all three SINTER calls in a single batch.

In a tuned-VM run, the bruteforce mode lands at `~18 ms` p50 decision-path
and the tightened mode lands at `~4.3 ms`. Most of that gap is the
sequential round-trip cost on the bruteforce side.

The candidate set is still ~50 ads at this stage — narrower than the 2500
of `full_realtime`, but the bid path now has to fetch each candidate's
metadata, evaluate the full per-campaign rules, and rerank. Notebook 4
removes most of *that* cost by doing the candidate generation offline.
